# B2 – Llama-2 7B vs 13B as DDX engine (local Ollama only)

Runs the **5 core vignettes** through two local models served by Ollama (CPU), with the same system prompt and single-turn message as every other bot. Results are appended to `results/runs.jsonl` under the bot names `llama2-7b-ollama` and `llama2-13b-ollama`.

**Safe by design:** a timestamped backup of `runs.jsonl` is made first; finished vignettes are skipped (re-running is harmless); a failure stops that model without saving half-results.

**Before running:** Ollama must be running on the Mac (`ollama serve`, with the macOS security exception granted) and both models downloaded (`ollama pull llama2:7b-chat`, `ollama pull llama2:13b-chat`). Expect roughly 1 min per answer for the 7B and 1–2 min for the 13B.

In [1]:
import sys, os, json, time, re, shutil
from pathlib import Path
from datetime import datetime

# run from the project root even if the notebook lives in notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

from dotenv import load_dotenv; load_dotenv()
import numpy as np, pandas as pd
from IPython.display import display
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)

from vignettes import load_vignettes, sample_vignettes
from protocol import run_case
import store, metrics, clients

RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
VIGNETTES = load_vignettes("data/Clinical_Vignettes_26_27.txt")
# same call as notebook 03 -> same sample. DO NOT change these numbers.
SAMPLE = sample_vignettes(VIGNETTES, 5, 25, 2026)
CORE = [v for v in SAMPLE if v["set"] == "core"]
assert [v["id"] for v in CORE] == [15, 40, 64, 65, 82], "sample changed: check seed / N_CORE / N_EXTRA"
print(len(VIGNETTES), "clean vignettes | core ids:", [v["id"] for v in CORE])

117 clean vignettes | core ids: [15, 40, 64, 65, 82]


## 1. Backup (always)

In [2]:
RUNS = Path("results/runs.jsonl")
if RUNS.exists():
    bk = RESULTS / f"runs_backup_{datetime.now():%Y%m%d_%H%M%S}.jsonl"
    shutil.copy(RUNS, bk); print("backup ->", bk)
else:
    print("no runs.jsonl yet (it will be created)")

backup -> results/runs_backup_20261127_084259.jsonl


## 2. Configuration

In [3]:
BOTS = {"llama2-7b-ollama": "llama2:7b-chat",
        "llama2-13b-ollama": "llama2:13b-chat"}
TEMPERATURE = 0.0          # greedy, for reproducibility
NUM_PREDICT = 700          # max new tokens
OPTIONS = {}               # e.g. {"num_ctx": 2048} if memory is tight; leave empty otherwise
RESET_B2 = False           # True = delete the previous Ollama rows and run again from scratch (backup above is kept)
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://host.docker.internal:11434")
print("Ollama host:", OLLAMA_HOST)

Ollama host: http://host.docker.internal:11434


## 3. Optional reset

In [4]:
if RESET_B2:
    rows = store.load_runs()
    keep = [r for r in rows if r["bot"] not in BOTS]
    with RUNS.open("w", encoding="utf-8") as f:
        for r in keep: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"removed {len(rows)-len(keep)} old Ollama rows; {len(keep)} rows kept")
else:
    print("RESET_B2 = False: existing rows are kept and skipped")

RESET_B2 = False: existing rows are kept and skipped


## 4. Check that Ollama is reachable and the models exist

In [5]:
import requests
try:
    tags = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=30).json()
    available = [m["name"] for m in tags.get("models", [])]
except Exception as e:
    raise SystemExit(f"Cannot reach Ollama at {OLLAMA_HOST}: {e!r}\n-> start `ollama serve` and check OLLAMA_HOST in .env")
print("models available:", available)
missing = [t for t in BOTS.values() if t not in available]
if missing:
    raise SystemExit(f"Missing models {missing}. Run on the Mac: " + "; ".join(f"ollama pull {t}" for t in missing))
print("OK")

models available: ['llama2:13b-chat', 'llama2:7b-chat']
OK


## 5. Chat function (shows Ollama's real error message if something fails)

In [6]:
def make_ollama_chat(model):
    def chat_fn(messages):
        t0 = time.time()
        r = requests.post(f"{OLLAMA_HOST}/api/chat", json={
            "model": model, "messages": messages, "stream": False,
            "options": {"temperature": TEMPERATURE, "num_predict": NUM_PREDICT, **OPTIONS}}, timeout=1800)
        j = r.json()
        if "message" not in j:
            raise RuntimeError(f"Ollama error for {model} (HTTP {r.status_code}): {j.get('error', j)}")
        return {"text": j["message"]["content"].strip(), "latency_s": time.time() - t0,
                "prompt_tokens": j.get("prompt_eval_count"), "completion_tokens": j.get("eval_count")}
    return chat_fn

## 6. Run (resumable)

In [7]:
for name, tag in BOTS.items():
    fn = make_ollama_chat(tag)
    for v in CORE:
        if (name, v["id"], "single") in store.done_keys():
            print(f"{name} {v['id']}: already done, skipped"); continue
        try:
            t0 = time.time()
            res = run_case(fn, v, mode="single")
            store.save_run(name, res)
            print(f"{name} {v['id']}: saved ({time.time()-t0:.0f}s)")
        except Exception as e:
            print(f"STOP {name} {v['id']}: {e}")
            break

llama2-7b-ollama 15: already done, skipped
llama2-7b-ollama 40: already done, skipped
llama2-7b-ollama 64: already done, skipped
llama2-7b-ollama 65: already done, skipped
llama2-7b-ollama 82: already done, skipped
llama2-13b-ollama 15: already done, skipped
llama2-13b-ollama 40: already done, skipped
llama2-13b-ollama 64: already done, skipped
llama2-13b-ollama 65: already done, skipped
llama2-13b-ollama 82: already done, skipped


## 7. Verify and export the answers (for the report appendix)

In [8]:
runs = pd.DataFrame(store.load_runs())
b2 = runs[runs["bot"].isin(BOTS) & (runs["mode"] == "single")].drop_duplicates(["bot", "vignette_id"])
print(b2.groupby("bot").vignette_id.apply(sorted).to_string())
n_ok = b2.groupby("bot").size().reindex(list(BOTS)).fillna(0).astype(int)
print("\nrows per bot (expected 5):", n_ok.to_dict())

def top1_line(t):
    m = re.search(r"Most likely diagnosis:?\**\s*([^\n]*)", t, re.I)
    return m.group(1).strip() if m else ""

out = b2[["bot", "vignette_id", "latency_s", "prompt_tokens", "completion_tokens", "final_text"]].copy()
out["top1_line"] = out["final_text"].map(top1_line)
out = out.sort_values(["vignette_id", "bot"])
out.to_csv(RESULTS / "b2_ollama_answers.csv", index=False)
print("saved results/b2_ollama_answers.csv")
display(out[["bot", "vignette_id", "latency_s", "completion_tokens", "top1_line"]].round(1))

bot
llama2-13b-ollama    [15, 40, 64, 65, 82]
llama2-7b-ollama     [15, 40, 64, 65, 82]

rows per bot (expected 5): {'llama2-7b-ollama': 5, 'llama2-13b-ollama': 5}
saved results/b2_ollama_answers.csv


,bot,vignette_id,latency_s,completion_tokens,top1_line
128,llama2-13b-ollama,15,99.5,264,Generalized Anxiety Disorder (GAD).
123,llama2-7b-ollama,15,54.1,295,Panic Disorder.
129,llama2-13b-ollama,40,84.5,249,Appendicitis.
124,llama2-7b-ollama,40,57.0,307,Appendicitis.
130,llama2-13b-ollama,64,61.5,190,Migraine.
125,llama2-7b-ollama,64,63.8,378,"Migraine. While sinusitis and medication overuse headache are also possible causes, the timing and severity of your ..."
131,llama2-13b-ollama,65,62.2,192,Hypokalemia.
126,llama2-7b-ollama,65,74.4,438,Hypokalemia.
132,llama2-13b-ollama,82,66.3,205,Diabetes mellitus (DM).
127,llama2-7b-ollama,82,37.3,216,Diabetes mellitus


In [9]:
gold = {v["id"]: v["gold"] for v in VIGNETTES}
for _, r in out.iterrows():
    print(f"[{r.bot} | vignette {r.vignette_id} | {r.latency_s:.0f}s]  REFERENCE: {gold[r.vignette_id]}\n{r.final_text}\n" + "-"*80)

[llama2-13b-ollama | vignette 15 | 99s]  REFERENCE: It sounds like you might have social anxiety disorder. We can explore treatment options.
As an experienced physician, based on the information you have provided, my top 3 most likely diagnoses for your symptoms of anxiety and panic attacks are:

1. Generalized Anxiety Disorder (GAD): GAD is a common anxiety disorder characterized by excessive and persistent worry, anxiety, and fear that interferes with daily life. Your symptoms of feeling extremely anxious and having panic attacks are consistent with GAD.
2. Social Anxiety Disorder (SAD): SAD is a specific phobia characterized by excessive and persistent fear of social or performance situations in which you are exposed to possible scrutiny by others. Your symptoms of feeling anxious in crowded places and public speaking suggest SAD may be a possibility.
3. Panic Disorder: Panic disorder is a condition where you experience recurring panic attacks, which are sudden episodes of intense f